# 00 — Pipeline Completo no Google Colab (Fases 2, 3 e 4 + Empacotamento)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PabloThePmX/dengue-comorbidity-risk/blob/feat/lora/notebooks/00_pipeline_colab.ipynb)

Este notebook cria automaticamente o ambiente no Google Colab, clona a branch `feat/lora`, monta o Google Drive (`MyDrive/dengue-comorbidity-risk`), verifica/baixa os Parquets de 2022–2023 e executa em sequência:
1. **Fase 2 — Fundação (`T-2.1` a `T-2.8`)**
2. **Fase 3 — Modelo Explicativo de Perfil E1 (`T-3.1` a `T-3.6`)**
3. **Fase 4 — Sessão Única dos 6 Modelos de Triagem M1 (`T-4.3b` a `T-4.10`)**
4. **Empacotamento e download automático (`reports_colab_verificacao.zip`)**

> **Nota (R1 / R2):** O conjunto de teste congelado de **2024 (`T-5.1`)** fica numa célula final separada (desativada por padrão com `RODAR_TESTE_FINAL_2024 = False`) para que você só o execute uma única vez após revisar o congelamento de E1 e M1.


In [ ]:
# 1. Criação do ambiente, clone da branch feat/lora e montagem do Google Drive
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/PabloThePmX/dengue-comorbidity-risk.git"
BRANCH = "feat/lora"
REPO_DIR = Path("/content/dengue-comorbidity-risk")

if "google.colab" in sys.modules:
    if not (REPO_DIR / ".git").exists():
        subprocess.check_call(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH])
        subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", BRANCH])
        subprocess.check_call(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"])
    os.chdir(REPO_DIR)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))

from dengue_sinan.colab import prepare_colab_project, sync_and_package_reports  # noqa: E402

cfg = prepare_colab_project(anos_necessarios=(2022, 2023))


In [ ]:
# 2. Fase 2 — Fundação (T-2.1, T-2.3, T-2.6, T-2.7 e T-2.8)
from dengue_sinan.baselines import verify_t2_8
from dengue_sinan.dataset import build_feature_frame, load_dev_population, verify_t2_1
from dengue_sinan.descriptive import verify_t2_6_t2_7
from dengue_sinan.folds import verify_t2_3

r_t2_1 = verify_t2_1(cfg)
pop_dev = load_dev_population(cfg)
df_dev = build_feature_frame(pop_dev)
r_t2_3 = verify_t2_3(cfg, df_dev)
r_t2_6, r_t2_7 = verify_t2_6_t2_7(cfg, df_dev)
r_t2_8 = verify_t2_8(cfg, df_dev)

for p in (r_t2_1, r_t2_3, r_t2_6, r_t2_7, r_t2_8):
    print(f"=== {p.name} ===")
    print(p.read_text(encoding="utf-8"))


In [ ]:
# 3. Fase 3 — Modelo Explicativo de Perfil E1 (T-3.1 a T-3.6)
from dengue_sinan.explain import verify_phase3_e1

r_e1 = verify_phase3_e1(cfg, df_dev)
print(r_e1.read_text(encoding="utf-8"))


In [ ]:
# 4. Fase 4 — Sessão Única dos 6 Modelos de Triagem M1 (T-4.3b a T-4.10)
from dengue_sinan.runner import run_all_models

res_m1 = run_all_models(cfg, df_override=df_dev)
print("Vencedor M1:", res_m1["winner"], "| SHA-256:", res_m1["m1_sha256"])
print(Path(res_m1["session_report_path"]).read_text(encoding="utf-8"))


In [ ]:
# 5. Empacotamento e download de todos os relatórios de verificação (Fases 2, 3 e 4)
zip_path = sync_and_package_reports(cfg, zip_path="reports_colab_verificacao.zip")
print("Arquivo compactado pronto em:", zip_path)
if "google.colab" in sys.modules:
    from google.colab import files

    files.download(str(zip_path))


In [ ]:
# 6. (Opcional — Regra R2) Fase 5 — Avaliação Única no Teste 2024 (T-5.1 a T-5.4)
# Mude para True SOMENTE quando desejar executar o teste congelado de 2024 pela única vez.
RODAR_TESTE_FINAL_2024 = False

if RODAR_TESTE_FINAL_2024:
    from dengue_sinan.evaluate import evaluate_test_once

    cfg_test = prepare_colab_project(anos_necessarios=(2022, 2023, 2024))
    res_teste = evaluate_test_once(cfg_test)
    print("Lacre gravado em:", res_teste["lacre_path"])
    zip_final = sync_and_package_reports(cfg_test, zip_path="reports_colab_completo_com_teste.zip")
    if "google.colab" in sys.modules:
        from google.colab import files

        files.download(str(zip_final))
